# Stage 3 - Phat hien cong dong & tien hoa cong dong trong mang hashtag TikTok dong

**Chuan bi:** tao Kaggle Dataset gom 2 file `video_hashtag_time.csv` + `stage3_dynamic_communities.py`, Add vao notebook.
Settings: **Internet = On** (de pip install). Accelerator: **None/CPU la du** (igraph/leidenalg chi chay CPU, GPU khong duoc dung).

| Buoc | Y nghia |
|---|---|
| TRAIN | Chon `gamma` (resolution) va `omega` (lien ket giua snapshot) tren 70% snapshot dau |
| EVALUATE | Chay 4 phuong phap tren toan bo snapshot, danh gia rieng train/test |

In [ ]:
!pip install -q igraph leidenalg

In [ ]:
import glob, os, shutil, sys
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50); pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 120)

csv = glob.glob('/kaggle/input/**/video_hashtag_time.csv', recursive=True)
py = glob.glob('/kaggle/input/**/stage3_dynamic_communities.py', recursive=True)
assert csv and py, 'Chua Add dataset chua video_hashtag_time.csv va stage3_dynamic_communities.py'
DATA = csv[0]
WORK = '/kaggle/working'
OUT = f'{WORK}/stage3_output'
shutil.copy(py[0], WORK)
os.chdir(WORK)
print('DATA =', DATA)
print('OUT  =', OUT)

## 1. TRAIN - chon tham so tren snapshot train
Tieu chi: `score = 0.5 * (modularity_t + predictive_modularity_{t->t+1})` (chat luong hien tai + kha nang tong quat sang snapshot ke tiep).

In [ ]:
!python stage3_dynamic_communities.py tune --input {DATA} --output {OUT}
display(pd.read_csv(f'{OUT}/tuning.csv').round(4))
print(open(f'{OUT}/tuning_result.json', encoding='utf-8').read())

## 2. EVALUATE - chay 4 phuong phap tren toan bo snapshot

In [ ]:
!python stage3_dynamic_communities.py run --input {DATA} --output {OUT}
display(pd.read_csv(f'{OUT}/snapshots.csv'))
comp = pd.read_csv(f'{OUT}/method_comparison.csv')
comp.round(4)

In [ ]:
met = pd.read_csv(f'{OUT}/snapshot_metrics.csv')
panels = [('modularity', 'Modularity Q'), ('nmi_prev', 'NMI voi snapshot truoc (on dinh)'),
          ('pred_modularity_next', 'Q cua P_t tren G_t+1 (du bao)'), ('n_communities_tracked', 'So cong dong (>= min_size)')]
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
for ax, (col, title) in zip(axes.ravel(), panels):
    for m, d in met.groupby('method', sort=False):
        ax.plot(d['snapshot'], d[col], marker='o', label=m)
    split_at = met[met['split'] == 'test']['snapshot'].min()
    ax.axvline(split_at, color='gray', ls='--', lw=1)
    ax.set_title(title); ax.tick_params(axis='x', rotation=45)
axes[0, 0].legend()
plt.tight_layout(); plt.savefig(f'{OUT}/fig_metrics_over_time.png', dpi=150); plt.show()

In [ ]:
ev = pd.read_csv(f'{OUT}/events.csv')
ev = ev[ev['event'].isin(['birth', 'death', 'merge', 'split', 'grow', 'shrink'])]
methods = ev['method'].unique()
fig, axes = plt.subplots(1, len(methods), figsize=(5 * len(methods), 4), sharey=True)
for ax, m in zip(axes, methods):
    tab = ev[ev['method'] == m].pivot_table(index='snapshot', columns='event', values='dyn_id', aggfunc='count', fill_value=0)
    tab.plot(kind='bar', stacked=True, ax=ax, title=m)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_events.png', dpi=150); plt.show()

In [ ]:
# Su kien lon nhat (theo kich thuoc) cua 2 phuong phap dong
for m in ['leiden_warm', 'leiden_multislice']:
    print('=' * 30, m)
    display(ev[ev['method'] == m].sort_values('size', ascending=False)
            .groupby('event').head(5)[['snapshot', 'event', 'dyn_id', 'size', 'related', 'top_hashtags']])

In [ ]:
# Cong dong dong song lau nhat (leiden_multislice)
tl = pd.read_csv(f'{OUT}/community_timeline.csv')
t = tl[tl['method'] == 'leiden_multislice']
life = t.groupby('dyn_id')['t'].nunique().sort_values(ascending=False)
for d in life.index[:8]:
    print(f'\n--- dyn_id={d}, song {life[d]} snapshot')
    display(t[t['dyn_id'] == d][['snapshot', 'size', 'share', 'event', 'jaccard', 'top_hashtags']])

## 3. (Tuy chon) Kiem tra do nhay - can bang so video moi snapshot
Du lieu crawl lech ve thang gan day (thang 10 nhieu video hon thang 1 ~10 lan). Chay lai voi toi da 450 video/snapshot de kiem tra ket luan co phu thuoc vao kich thuoc mau khong.

In [ ]:
import json
best = json.load(open(f'{OUT}/tuning_result.json'))
OUT_B = f'{WORK}/stage3_output_balanced'
G, W = best['gamma'], best['omega']
!python stage3_dynamic_communities.py run --input {DATA} --output {OUT_B} --max-videos 450 --gamma {G} --omega {W}
pd.read_csv(f'{OUT_B}/method_comparison.csv').query("split == 'all'").round(4)

In [ ]:
# Nen ket qua de tai ve (Output -> stage3_results.zip)
shutil.make_archive(f'{WORK}/stage3_results', 'zip', WORK, 'stage3_output')
if os.path.exists(f'{WORK}/stage3_output_balanced'):
    shutil.make_archive(f'{WORK}/stage3_results_balanced', 'zip', WORK, 'stage3_output_balanced')
print(os.listdir(WORK))